# 03 · Core grid (H1, H2b): S0, S1, C0, C1 × 5 budgets × 5 seeds

**Runtime → Change runtime type → T4 GPU.** Then **Runtime → Run all**.

The queue runs seed by seed so that an interrupted session still leaves complete seeds:
for each seed, SSL pretraining (P_ecg mid), then S1 and C1 at 5/10/25/50/100%, then S0 and C0.
Finished runs are skipped and interrupted runs resume from their Drive checkpoint, so after a disconnect just
**Run all** again. Estimated ≈ 80 min per seed on a T4 (D31); seed 0 is partly done.

The exploratory 1% budget runs last (RQ v2 D10). Do not use *File → Save a copy in GitHub*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, zipfile, json
PROJECT = '/content/drive/MyDrive/robust-ecg-ssl'
CACHE_DRIVE, RUNS, SSL = f'{PROJECT}/cache', f'{PROJECT}/runs', f'{PROJECT}/ssl'
CACHE = '/content/cache'                      # local copy: much faster reads than Drive
REPO = '/content/robust-ecg-ssl'
os.makedirs(SSL, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q https://github.com/Salako07/robust-ecg-ssl.git {REPO}
!git -C {REPO} pull -q && git -C {REPO} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# copy the cache to local disk once per session (~1-3 min). Retries if the Drive mount drops
# ("Transport endpoint is not connected"): remounts Drive and resumes; finished files are skipped.
import sys; sys.path.insert(0, f'{REPO}/src')
from robust_ecg.colab_utils import copy_dir_with_retry
copy_dir_with_retry(CACHE_DRIVE, CACHE, remount=lambda: drive.mount('/content/drive', force_remount=True))
!ls -la {CACHE}

In [ ]:
import subprocess, time
SEEDS, BUDGETS = [0, 1, 2, 3, 4], [0.05, 0.1, 0.25, 0.5, 1.0]
def ssl_run(seed): return f'{SSL}/SSL_ecg-mid_s{seed}'
queue = []
for seed in SEEDS:
    queue.append(('SSL', f'SSL_ecg-mid_s{seed}', ['scripts/pretrain_simclr.py', '--policy', 'ecg', '--strength', 'mid',
                  '--seed', str(seed), '--cache', CACHE, '--runs', SSL, '--workers', '2']))
    for arms in (('S1', 'C1'), ('S0', 'C0')):
        for b in BUDGETS:
            for arm in arms:
                args = ['scripts/train.py', '--arm', arm, '--budget', str(b), '--seed', str(seed),
                        '--cache', CACHE, '--runs', RUNS, '--workers', '2']
                if arm.startswith('C'):
                    args += ['--pretrained', f'{ssl_run(seed)}/encoder.pt']
                queue.append((arm, f'{arm}_b{b:g}_s{seed}_lik0', args))
for seed in SEEDS:                                   # exploratory 1% budget, last
    for arm in ('S1', 'C1', 'S0', 'C0'):
        args = ['scripts/train.py', '--arm', arm, '--budget', '0.01', '--seed', str(seed),
                '--cache', CACHE, '--runs', RUNS, '--workers', '2']
        if arm.startswith('C'):
            args += ['--pretrained', f'{ssl_run(seed)}/encoder.pt']
        queue.append((arm, f'{arm}_b0.01_s{seed}_lik0', args))

def is_done(kind, rid):
    return os.path.exists(f'{SSL if kind == "SSL" else RUNS}/{rid}/done.json')
todo = [q for q in queue if not is_done(q[0], q[1])]
print(f'{len(queue)} runs in the grid, {len(queue) - len(todo)} finished, {len(todo)} to do')

## Run the queue
Prints one line per run (full per-step logs are in each run's `log.csv` on Drive). A failed run stops the queue
and prints its last output lines: **paste them into the chat**.

In [ ]:
for i, (kind, rid, args) in enumerate(todo, 1):
    t0 = time.time()
    p = subprocess.run(['python', f'{REPO}/{args[0]}', *args[1:]], capture_output=True, text=True)
    if p.returncode != 0:
        print(f'FAILED {rid}\n' + '\n'.join((p.stdout + p.stderr).splitlines()[-30:]))
        break
    last = p.stdout.strip().splitlines()[-1] if p.stdout.strip() else ''
    print(f'[{i}/{len(todo)}] {rid}  {(time.time() - t0) / 60:.1f} min  {last[:160]}', flush=True)

## Progress table
Test macro-AUROC (71 statements) and SPH E3 macro-AUROC per arm and budget, mean over finished seeds.
**Descriptive only.** The pre-registered analysis (paired patient-level bootstrap, Holm) is a separate script.

In [ ]:
import pandas as pd
reg = pd.read_csv(f'{RUNS}/registry.csv').drop_duplicates('run_id', keep='last')
reg = reg[reg.min_likelihood == 0]
for col in ('test_macro_auroc', 'sph_e3_macro_auroc', 'e3_degradation'):
    t = reg.pivot_table(index='budget', columns='arm', values=col, aggfunc=['mean', 'count'])
    print(col); print(t.round(4).to_string()); print()

## Export results for the repository
Download `results_export.zip` from Drive and attach it in the chat after each session.

In [ ]:
out = f'{PROJECT}/results_export.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, arc in ((RUNS, 'runs'), (SSL, 'ssl')):
        for f in glob.glob(f'{root}/*/*'):
            if f.endswith(('config.json', 'log.csv', 'done.json', 'predictions.npz')) and \
               os.path.exists(os.path.join(os.path.dirname(f), 'done.json')):
                z.write(f, os.path.join(arc, os.path.relpath(f, root)))
    for reg, arc in ((f'{RUNS}/registry.csv', 'runs/registry.csv'), (f'{SSL}/registry_ssl.csv', 'ssl/registry_ssl.csv')):
        if os.path.exists(reg):
            z.write(reg, arc)
print(out, round(os.path.getsize(out) / 1e6, 2), 'MB')
print(zipfile.ZipFile(out).namelist())